In [1]:
import sys
from pathlib import Path
import re
import pymupdf
import pymupdf4llm
from langchain_core.documents import Document
import json

In [4]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
DATA = ROOT_DIR / 'data'
from src.config import BANQUES

In [5]:
BANQUES

{'bnpparibas': ('BNP Paribas', 'fr'),
 'ca': ('Credit Agricole', 'fr'),
 'caex': ('Credit Agricole', 'fr'),
 'ubs': ('UBS', 'en')}

In [6]:
pdf = pymupdf4llm.to_markdown(DATA /'raw/caex_2025.pdf',
                                page_chunks=True,
                               show_progress=True)

Parsing 58 pages of '/home/yanis/projects/rapFin_assistant/data/raw/caex_2025.pdf'...
[========================================]
Generating markdown text...
[========================================]


In [7]:
pdf[0] # dict de page 1
pdf[0].keys() # dict_keys(['metadata', 'toc_items', 'page_boxes', 'text'])
#pdf[0]['page_boxes'][0]# Retourne boites (dictionnaire), compose de class:'picture','text','section-header' (on garde text) et 'pos', un tuple correspondant a la position du texte dans pages[0]['text']

dict_keys(['metadata', 'toc_items', 'page_boxes', 'text'])

In [8]:
pdf[20]['metadata']['page_number']

21

In [9]:
def extract(page):
    """Genere (class,text) d'une boite pour une page """
    for ligne in page["page_boxes"]:
        debut,fin = ligne['pos']
        yield ligne['class'], page['text'][debut:fin]

In [10]:
for ligne in extract(pdf[10]):
    print(ligne[1])



<!-- Start of picture text -->
Comptes consolidés du groupe Crédit Agricole S.A.   – 31 décembre 2025<br><!-- End of picture text -->


#### **RÉSULTAT NET ET GAINS ET PERTES COMPTABLISÉS DIRECTEMENT EN CAPITAUX PROPRES** 


|_(en millions d'euros)_|**Notes**|**31/12/2025**|**31/12/2024**|
|---|---|---|---|
|**Résultat net**||**8 053**|**8 154 **|
|Gains etpertes actuariels suravantages postemploi|4.12|49|39|
|Gains et pertes sur passifs financiers attribuables aux variations du risque de<br>||||
|créditpropre|4.12|(234)|(365)|
|Gains et pertes sur instruments de capitaux propres comptabilisés en<br>||||
|capitauxpropresnon recyclables|4.12|276|117|
|Produits financiers ou charges financières d'assurance ventilés en capitaux<br>propresnon recyclables|4.12|(166)|(82)|
|**Gains et pertes avant impôt comptabilisés directement en capitaux propres**<br>**non recyclables hors entreprises mises en équivalence**|**4.12**|**(76)**|**(291)**|
|**Gains et pertes avant impôt comptabilisés direct

In [11]:
for ligne in extract(pdf[40]):
    print(ligne)

('picture', '\n\n<!-- Start of picture text -->\nComptes consolidés du groupe Crédit Agricole S.A.   – 31 décembre 2025<br><!-- End of picture text -->\n\n')
('section-header', '###### **_Risques fiscaux_** \n\n')
('text', 'Les risques fiscaux portant sur l’impôt sur le résultat donnent lieu à la comptabilisation d’une créance ou d’une dette d’impôt lorsque la probabilité de recevoir l’actif ou de payer le passif est jugée plus probable qu’improbable. Ces risques sont par ailleurs pris en compte dans l’évaluation des actifs et passifs d’impôts courants et différés. \n\n')
('text', 'L’interprétation IFRIC 23 portant sur l’évaluation des positions fiscales incertaines s’applique dès lors qu’une entité a identifié une ou des incertitudes à propos de positions fiscales prises concernant ses impôts sur le résultat. Elle apporte également des précisions sur leurs estimations : \n\n')
('list-item', '- l’analyse doit être fondée sur une détection à 100 % par l’administration fiscale ; \n\n')
(

In [12]:
def clean_text(texte,langue):
    """Nettoie texte en retirant balise html apres conversion md, et les soulignes/gras/italique de md et bug genre fi gures"""
    texte = re.sub(r"<br\s*/?>", " ", texte)
    texte = re.sub(r"</?u>|\*{1,3}|(?<!\w)_+|_+(?!\w)", "", texte)
    if langue == "en":
        texte = re.sub(r"(fi|fl) (?=[a-z])", r"\1", texte)
    return re.sub(r"\n{3,}", "\n\n", texte).strip()

In [13]:
for ligne in extract(pdf[0]):
    if ligne[0] == 'text':
        print(clean_text(ligne[1],'fr'))

Comptes consolidés du groupe Crédit Agricole S.A.
31 décembre 2025
Arrêtés par le Conseil d’administration de Crédit Agricole S.A. en date du 3 février 2026 et soumis à l’approbation de l’Assemblée générale ordinaire en date du 20 mai 2026
VERSION NON AUDITÉE


In [14]:
def load_report(path):
    """ Renvoie une liste des pages du pdf dans path sous type Document (type loader LangChain) """
    liste_pages = []
    cle_banque, annee = path.stem.rsplit('_',1)
    banque, langue = BANQUES[cle_banque]
    def garder(classe):
        if classe in ('picture', 'page-footer'):
            return False
        return True
    pdf = pymupdf4llm.to_markdown(path, page_chunks=True,show_progress=True)
    for page in pdf:
        texte_page = "\n".join(texte_ligne for classe,texte_ligne in extract(page) if garder(classe))
        doc = Document(
            page_content = clean_text(texte_page,langue),
            metadata = {
                "source": path.name ,
                "banque": banque,
                "annee": int(annee),
                "langue": langue,
                "page": page['metadata']['page_number']
            })
        liste_pages.append(doc)
    return liste_pages
        

In [15]:
ca = load_report(DATA / 'raw/caex_2025.pdf')

Parsing 58 pages of '/home/yanis/projects/rapFin_assistant/data/raw/caex_2025.pdf'...
[========================================]
Generating markdown text...
[========================================]


In [16]:
def load_reports(root_path):
    all_pages = []
    for pdf in sorted(root_path.glob('*.pdf')):
        pages = load_report(pdf)
        all_pages.extend(pages)
        print(f'{len(pages)} pages du document {pdf.name} ont ete lues.')
    return all_pages

In [17]:
def save_pages(liste_docs,path):
    """Sauvegarde dans fichiers JSON les documents """
    with path.open('w', encoding="utf-8") as f:
        for doc in liste_docs:
            d = {"page_content": doc.page_content,
                 "metadata": doc.metadata}
            f.write(json.dumps(d) + '\n')

In [20]:
def dump_data(in_path,out_path):
    for pdf in sorted(in_path.glob('*.pdf')):
        # Rajouter verif si deja dans out_path
        print(pdf)
        if pdf.stem not in [f.stem for f in sorted(out_path.glob('*.jsonl'))]:
            doc = load_report(pdf)
            name = pdf.with_suffix('.jsonl').name
            save_pages(doc, out_path / name )

In [22]:
dump_data(DATA / 'raw/', DATA / 'interim')

/home/yanis/projects/rapFin_assistant/data/raw/bnpparibas_2025.pdf
/home/yanis/projects/rapFin_assistant/data/raw/ca_2025.pdf
/home/yanis/projects/rapFin_assistant/data/raw/caex_2025.pdf
Parsing 58 pages of '/home/yanis/projects/rapFin_assistant/data/raw/caex_2025.pdf'...
[========================================]
Generating markdown text...
[========================================]
/home/yanis/projects/rapFin_assistant/data/raw/ubs_2025.pdf
